In [3]:
import sqlite3
import pandas as pd
import torch
from transformers import pipeline
import os

# 1. Configurazione Accelerazione Hardware (Apple M2)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# 2. Estrazione testi e pulizia
db_path = '../data/document_analysis.db'
conn = sqlite3.connect(db_path)

# Query aggiornata: JOIN tra rdf_triples e documents
query = """
SELECT 
    COALESCE(ea1.canonical_name, t.actor) AS source,
    COALESCE(ea2.canonical_name, t.target) AS target,
    d.paragraph_summary
FROM rdf_triples t
JOIN documents d ON t.doc_id = d.doc_id
LEFT JOIN entity_aliases ea1 ON t.actor = ea1.original_name
LEFT JOIN entity_aliases ea2 ON t.target = ea2.original_name
WHERE d.paragraph_summary IS NOT NULL AND d.paragraph_summary != ''
"""
df_texts = pd.read_sql_query(query, conn)
conn.close()

# Pulizia valori nulli e self-loop
df_texts = df_texts.dropna(subset=['source', 'target'])
df_texts = df_texts[df_texts['source'] != df_texts['target']]

# Applichiamo lo stesso filtro di pulizia usato nella Network Analysis
extraction_artifacts = [
    'author', 'communication', 'association', 'lawsuit', 'document', 
    'email', 'letter', 'meeting', 'payment', 'none', 'unknown'
]
df_texts = df_texts[
    (~df_texts['source'].str.lower().isin(extraction_artifacts)) & 
    (~df_texts['target'].str.lower().isin(extraction_artifacts))
].copy()

# 3. Setup Modello RoBERTa per la Sentiment Analysis
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model=model_name, 
    device=device, 
    truncation=True, 
    max_length=512
)

# 4. Inferenza (Classificazione Sentiment sui riassunti)
# Convertiamo in lista per l'elaborazione in batch sulla GPU del Mac
texts = df_texts['paragraph_summary'].tolist()
print(f"🔄 Inizio analisi sentiment su {len(texts)} relazioni. Potrebbe volerci qualche minuto...")

results = sentiment_analyzer(texts, batch_size=16)

# 5. Mappatura dei risultati
df_texts['sentiment_label'] = [res['label'] for res in results]
df_texts['sentiment_score'] = [res['score'] for res in results]

# Conversione in punteggio numerico: negative = -1, neutral = 0, positive = 1
sentiment_map = {'negative': -1, 'neutral': 0, 'positive': 1}
df_texts['sentiment_value'] = df_texts['sentiment_label'].map(sentiment_map)

# 6. Aggregazione: calcolo del peso emotivo medio per ogni arco
df_edges_sentiment = df_texts.groupby(['source', 'target']).agg(
    weight=('source', 'size'),
    avg_sentiment=('sentiment_value', 'mean')
).reset_index()

# 7. Esportazione finale per la Dashboard
output_dir = '../data/'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'edges_sentiment.csv')
df_edges_sentiment.to_csv(output_file, index=False)

print("✅ Sentiment Analysis completata e salvata!")
# Mostriamo le prime righe per confermare il successo
df_edges_sentiment.head(10)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 39667.58it/s]
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Inizio analisi sentiment su 105875 relazioni. Potrebbe volerci qualche minuto...
✅ Sentiment Analysis completata e salvata!


,source,target,weight,avg_sentiment
0,Edward Snowden,The Nation,1,0.0
1,/ronic Times,"""Quaaludes Ends""",1,0.0
2,10th Circuit Court,victim’s right to attend a trial,1,0.0
3,10thAmend,ooohouchburn,1,0.0
4,11 SLOANS CURVE LLC,SLOANS CURVE townhouse unit 11,1,0.0
5,111th Military Intelligence Group,Martin Luther King,1,0.0
6,11th Circuit,review of plea correspondence,1,0.0
7,14-year-old girl,Jeffrey Epstein,1,-1.0
8,163 countries,UNCAC,1,0.0
9,170141 Canada Inc.,Halcyon of Palm Beach Condo 1010,1,0.0
